# 1. Instalando as Bibliotecas

In [21]:
!pip -q install \
langchain \
langchain-community \
langchain-groq \
langchain-text-splitters \
faiss-cpu \
sentence-transformers \
pypdf \
python-dotenv

# 2. Importando as bibliotecas

In [22]:
import os

from google.colab import userdata

# Documentos
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Divisão de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Banco Vetorial
from langchain_community.vectorstores import FAISS

# Modelo Groq
from langchain_groq import ChatGroq

# Prompt
from langchain_core.prompts import ChatPromptTemplate

# Chains
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("✅ Bibliotecas carregadas.")

✅ Bibliotecas carregadas.


# 3. Clonar o repositório do GitHub

In [23]:
!rm -rf PortfolioAI

!git clone https://github.com/elissouza2023/PortfolioAI.git

BASE_PATH = "/content/PortfolioAI"

KNOWLEDGE_PATH = f"{BASE_PATH}/knowledge_base"

VECTOR_PATH = f"{BASE_PATH}/vector_store"

os.makedirs(VECTOR_PATH, exist_ok=True)

print("✅ Repositório clonado.")

Cloning into 'PortfolioAI'...
remote: Enumerating objects: 234, done.
remote: Total 234 (delta 0), reused 0 (delta 0), pack-reused 234 (from 1)
Receiving objects: 100% (234/234), 37.61 MiB | 24.04 MiB/s, done.
Resolving deltas: 100% (102/102), done.
✅ Repositório clonado.


# 4. Configurar API Key da Groq

In [24]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ API Key carregada.")

✅ API Key carregada.


# 5. Carregando documentos da pasta knowledge_base

In [25]:
loader = PyPDFDirectoryLoader(KNOWLEDGE_PATH)

documents = loader.load()

print(f"\n📄 Total de documentos: {len(documents)}")

for doc in documents:
    print(doc.metadata["source"])


📄 Total de documentos: 83
/content/PortfolioAI/knowledge_base/Curriculo_Elisangela_de_Souza resumido TI.pdf
/content/PortfolioAI/knowledge_base/Curriculo_Elisangela_de_Souza resumido TI.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/co

# 6. Dividindo os documentos em chunks

In [26]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=900,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ".",
        "!",
        "?",
        " "
    ]
)

texts = text_splitter.split_documents(documents)

print(f"✅ Chunks criados: {len(texts)}")

✅ Chunks criados: 186


# 7. Criando Embeddings

In [27]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# 8. Banco Vetorial

In [28]:
vector_store = FAISS.from_documents(

    texts,

    embeddings

)

vector_store.save_local(VECTOR_PATH)

print("✅ Banco vetorial criado.")

✅ Banco vetorial criado.


In [29]:
from google.colab import files
import os

# Compacta a pasta
!zip -r vector_store.zip /content/PortfolioAI/vector_store

# Faz o download
files.download("vector_store.zip")

updating: content/PortfolioAI/vector_store/ (stored 0%)
updating: content/PortfolioAI/vector_store/index.faiss (deflated 7%)
updating: content/PortfolioAI/vector_store/index.pkl (deflated 72%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 9. Modelo Groq

In [60]:
MODEL_NAME = "qwen/qwen3.6-27b"

llm = ChatGroq(

    model_name=MODEL_NAME,

    temperature=0.2,

    max_tokens=1200

)

print("✅ Modelo carregado.")

✅ Modelo carregado.


## 10. Prompt do PortfolioAI

In [61]:
system_prompt = """
Você é o PortfolioAI.

Seu objetivo é responder perguntas sobre Elisângela de Souza.

REGRAS IMPORTANTES

• Utilize EXCLUSIVAMENTE as informações presentes no contexto.

• Nunca invente experiências.

• Nunca complete informações por conta própria.

• Caso não exista resposta no contexto, diga:

"Não encontrei essa informação na minha base de conhecimento.
Caso deseje mais detalhes, recomendo entrar em contato diretamente com Elisângela."

• Sempre escreva de forma profissional, porém em primeira pessoal como em uma entrevista, lembre-se de que vocÊ é o meu asistente pessoal. Por exemplo diga sou uma profissional...

• Manetenha racioninio fluído, linguagem clara, tom entusiastico. Mantenha o tom de conversa, como em uma entrevista de emprego.

• Sempre responda em português.

• Quando possível organize a resposta em tópicos.

Contexto:

{context}
"""

prompt = ChatPromptTemplate.from_messages(

    [

        ("system", system_prompt),

        ("human", "{input}")

    ]

)

print("✅ Prompt criado.")

✅ Prompt criado.


# 11. Chain RAG

In [62]:
question_answer_chain = create_stuff_documents_chain(

    llm,

    prompt

)

retriever = vector_store.as_retriever(

    search_kwargs={

        "k":6

    }

)

rag_chain = create_retrieval_chain(

    retriever,

    question_answer_chain

)

print("✅ RAG criado.")

✅ RAG criado.


# 12. Função para perguntas

In [63]:
def perguntar(pergunta):

    resposta = rag_chain.invoke(

        {

            "input": pergunta

        }

    )

    print("="*80)

    print("PERGUNTA")

    print(pergunta)

    print()

    print("RESPOSTA")

    print(resposta["answer"])

    print()

    print("FONTES UTILIZADAS")

    fontes = set()

    for doc in resposta["context"]:

        fontes.add(

            os.path.basename(

                doc.metadata["source"]

            )

        )

    for fonte in sorted(fontes):

        print("•", fonte)

    print("="*80)

# 13. Testes

In [64]:
perguntar("Quem é a Elisângela?")

PERGUNTA
Quem é a Elisângela?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User asks: "Quem é a Elisângela?" (Who is Elisângela?)
   - Language: Portuguese
   - Context provided: Contains detailed information about Elisângela de Souza, including her background, education, career transition, location, and professional goals.

2.  **Identify Key Information from Context:**
   - Name: Elisângela de Souza
   - Location: Volta Redonda – RJ
   - Background: Combines solid operational and administrative experience in the industry with an active transition to Information Technology.
   - Education: Bachelor's in Administration, Postgraduate in Metallurgical Engineering, currently finishing Technology in Information Security.
   - Professional Goal: Work in IT, especially in Information Security, Technical Support/Cloud, Data Analysis, or projects combining administration and technology.
   - Contact/Links (optional but good to know): Email, LinkedIn, GitHub, 

In [65]:
perguntar("Qual é o seu objetivo profissional ?")

PERGUNTA
Qual é o seu objetivo profissional ?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Qual é o seu objetivo profissional ?" (What is your professional goal?)
   - **Language:** Portuguese
   - **Context Provided:** The context contains repeated paragraphs about Elisângela de Souza's professional journey, values, and current focus. Key phrases include:
     - "busco desenvolver soluções inteligentes, centradas nas pessoas e orientadas pela inovação."
     - "projetos atuais em Inteligência Artificial"
     - "baseadas em tecnologia e Inteligência Artificial."
     - Mentions of growth through complementary experiences, from operational tasks to current AI projects.
     - Values: transparency, feedback, responsibility, discipline, teamwork, continuous learning.

2.  **Identify Relevant Information in Context:**
   - The context explicitly states: "...na qual busco desenvolver soluções inteligentes, centradas nas pessoas e orientadas

In [66]:
perguntar("Quais projetos ela desenvolveu?")

PERGUNTA
Quais projetos ela desenvolveu?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Quais projetos ela desenvolveu?" (What projects did she develop?)
   - **Target:** Elisângela de Souza
   - **Language:** Portuguese

2.  **Scan Context for Keywords:**
   - Keywords: "projetos", "desenvolveu", "lista", "grupos", "Projetos Estratégicos"
   - Context mentions: "Este documento reúne os principais projetos desenvolvidos por Elisângela de Souza..."
   - Context mentions: "Os projetos foram organizados em três grupos. Projetos Estratégicos Projetos que representam diretamente minha atuação profissional e as áreas"
   - The context cuts off abruptly after "Projetos Estratégicos Projetos que representam diretamente minha atuação profissional e as áreas". It repeats some sections but doesn't actually list specific project names or details.

3.  **Evaluate Findings against Rules:**
   - Rule: "Utilize EXCLUSIVAMENTE as informações presentes no 

In [67]:
perguntar("Fale sobre o projeto PortfolioAI.")

PERGUNTA
Fale sobre o projeto PortfolioAI.

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Fale sobre o projeto PortfolioAI." (Tell me about the PortfolioAI project.)
   - **Language:** Portuguese
   - **Context Provided:** The context contains information about "PortfolioAI – Base de Conhecimento do RAG", its purpose, structure, and some philosophical/learning aspects related to Elisângela de Souza.

2.  **Extract Relevant Information from Context:**
   - *Name:* PortfolioAI – Base de Conhecimento do RAG
   - *Purpose/Goal:* Reúne os principais projetos desenvolvidos por Elisângela de Souza ao longo de sua trajetória de formação e transição para a área de Tecnologia da Informação. Apresenta contexto, problemas abordados, tecnologias empregadas e competências demonstradas. Permite responder perguntas sobre projetos, ML, UX, Segurança da Informação, Python, estruturação, etc.
   - *Structure:* Organizado em três grupos (Projetos Estratégico

In [68]:
perguntar("Quais competências técnicas ela possui?")

PERGUNTA
Quais competências técnicas ela possui?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Quais competências técnicas ela possui?" (What technical skills does she have?)
   - **Subject:** Elisângela de Souza
   - **Language:** Portuguese

2.  **Scan Context for Keywords:**
   - Keywords: "competências técnicas", "técnicas", "ferramentas", "conhecimentos", "Inteligência Artificial", "soluções"
   - Context mentions:
     - "Considero que competências técnicas e comportamentais são complementares."
     - "Enquanto o conhecimento técnico permite construir soluções..."
     - "Enquanto as competências técnicas representam os conhecimentos e ferramentas que utilizo..."
     - "...projetos atuais em Inteligência Artificial..."
     - "Utilizar Inteligência Artificial de forma ética, transparente e responsável."
   - **Crucial Observation:** The context *mentions* technical skills in a general/complementary sense and references "Inteligên

In [69]:
perguntar("Qual sua formação acadêmica?")

PERGUNTA
Qual sua formação acadêmica?

RESPOSTA

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "Qual sua formação acadêmica?" (What is your academic background?)
   - **Language:** Portuguese
   - **Context Provided:** The context contains repeated sections about "Aprendizado Contínuo" (Continuous Learning), "Síntese" (Synthesis), and "FORMAÇÃO ACADÊMICA" (Academic Background) for Elisângela de Souza. However, it *does not* list specific degrees, institutions, or dates. It only talks about the philosophy behind her academic background, continuous learning, certifications, courses, and how it integrates management, processes, technology, and innovation.

2.  **Check Constraints:**
   - Use EXCLUSIVELY information from the context.
   - Never invent experiences.
   - Never complete information on my own.
   - If not in context, say: "Não encontrei essa informação na minha base de conhecimento. Caso deseje mais detalhes, recomendo entrar em contato dir